# Partie 1 : Pourquoi InfluxDB ?

## SQL vs InfluxDB



In [1]:
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS
import MySQLdb

# Config InfluxDB
URL = "http://influxdb2:8086"
TOKEN = "MyInitialAdminToken0=="
ORG = "docs"
BUCKET = "home"
MEASUREMENT = "home"

# Config MySQL
MYSQL_HOST = "mysql"
MYSQL_PORT = 3306
MYSQL_USER = "student"
MYSQL_PASSWORD = "student"
MYSQL_DATABASE = "home_data"

# Client InfluxDB
client = InfluxDBClient(url=URL, token=TOKEN, org=ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)

# Client MySQL
mysql_conn = MySQLdb.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    passwd=MYSQL_PASSWORD,
    db=MYSQL_DATABASE,
    autocommit=False,
    connect_timeout=5,
)
mysql_cursor = mysql_conn.cursor()

In [2]:
import time

def run_sql_query(query, cursor):
  start = time.time()
  cursor.execute(query)
  result = cursor.fetchall()
  return time.time() - start


def run_influx_query(query, client):
  query_api = client.query_api()
  start = time.time()
  result = query_api.query(query)
  return time.time() - start

In [3]:
sql_query = """
SELECT
  DATE_FORMAT(time_ts, '%Y-%m-%d %H:00:00') AS hour,
  AVG(temp) AS avg_temp
FROM sensor_data
WHERE time_ts >= NOW() - INTERVAL 7 DAY
GROUP BY hour
ORDER BY hour;
"""

flux_query = '''
from(bucket: "home")
  |> range(start: -7d)
  |> filter(fn: (r) => r._measurement == "sensor_data")
  |> filter(fn: (r) => r._field == "temp")
  |> aggregateWindow(every: 1h, fn: mean, createEmpty: false)
'''

print("SQL time:", run_sql_query(sql_query, mysql_cursor))
print("InfluxDB time:", run_influx_query(flux_query, client))

SQL time: 0.033252716064453125
InfluxDB time: 0.01750969886779785


In [44]:
sql_query = """
SELECT
  DATE_FORMAT(time_ts, '%Y-%m-%d %H:00:00') AS hour,
  AVG(temp) AS avg_temp
FROM sensor_data
WHERE time_ts >= NOW() - INTERVAL 7 DAY
  AND room = 'A101'
GROUP BY hour
ORDER BY hour;
"""

flux_query = '''
from(bucket: "home")
  |> range(start: -7d)
  |> filter(fn: (r) => r._measurement == "sensor_data")
  |> filter(fn: (r) => r._field == "temp")
  |> filter(fn: (r) => r.room == "A101")
  |> aggregateWindow(every: 1h, fn: mean, createEmpty: false)
'''

print("SQL time:", run_sql_query(sql_query, mysql_cursor))
print("InfluxDB time:", run_influx_query(flux_query, client))

SQL time: 0.00731348991394043
InfluxDB time: 0.023580551147460938


In [45]:
sql_query = """
SELECT
  floor_name,
  DATE(time_ts) AS day,
  AVG(co2) AS avg_co2
FROM sensor_data
WHERE time_ts >= NOW() - INTERVAL 7 DAY
GROUP BY floor_name, day
ORDER BY floor_name, day;
"""

flux_query = '''
from(bucket: "home")
  |> range(start: -7d)
  |> filter(fn: (r) => r._measurement == "sensor_data")
  |> filter(fn: (r) => r._field == "co2")
  |> group(columns: ["floor"])
  |> aggregateWindow(every: 1d, fn: mean, createEmpty: false)
'''

print("SQL time:", run_sql_query(sql_query, mysql_cursor))
print("InfluxDB time:", run_influx_query(flux_query, client))

SQL time: 0.26329708099365234
InfluxDB time: 0.02868199348449707
